# Sampling Rate and Bit Depth: Quality and Storage Trade-offs in Digital Audio
Reproducible experiment notebook for the CEPAT manuscript.

**How to use (Google Colab)**
1. Run the *dataset* cell: it downloads open speech and music clips automatically into `/content/audio/<category>/*.wav` (no manual upload). To use your own material instead, skip that cell and place WAV files in one folder per category (the folder name is used as the content category).
2. Edit the CONFIG cell if needed (sampling rates, bit depths, trim, Welch window).
3. Run all cells. Results (CSV + PNG figures + environment log + data manifest) are written to `/content/results` (Colab) or `./content/results` next to the notebook (local run).

**Design.** Reference signal = clip at its native rate, mono, float64, peak-normalized.
* Series A - sampling rate only: reference -> fs_k -> back to native rate (no quantization).
* Series B - bit depth only: reference -> B-bit uniform quantizer at the native rate.
* Series C - full factorial fs_k x B: reference -> fs_k -> B-bit quantizer -> back to native rate.

Quality metrics are computed on the in-memory signals. WAV files are written only to measure real file size.
Nothing in this notebook contains pre-computed results.

In [ ]:
# %pip installs into the kernel that runs this notebook (works in Colab, Jupyter, VS Code/Kiro).
# Colab already ships numpy/scipy/pandas/matplotlib; a local kernel may not.
%pip install -q numpy scipy pandas soundfile matplotlib
# Optional: upload clips from your computer (or mount Google Drive and point AUDIO_DIR to it)
# from google.colab import files; files.upload()   # then move files into /content/audio/<category>/

## Audio material (automatic download)
* **Speech** - EARS dataset [Richter et al., Interspeech 2024], anechoic studio speech, 48 kHz, 32-bit float, mono, CC BY-NC 4.0. Only the selected WAV members are read from each speaker ZIP through HTTP range requests, so the 0.6 GB archives are not downloaded in full. Default: speakers p001-p004 (2 male, 2 female), 2 read sentences (regular style) each.
* **Music** - Musopen recordings hosted on Wikimedia Commons, 48 kHz, 24-bit, stereo FLAC, public domain (orchestra and string quartet). Default: 4 recordings, 2 non-overlapping excerpts each.

Every clip is cut to `CLIP_S` seconds and stored as WAV in its native sample format, so the reference keeps its native sampling rate and bit depth. `data_manifest.csv` records source URL, license, and offset for the Data Availability statement. Wikimedia rejects requests without an identifying User-Agent (HTTP 403) and throttles bulk traffic (HTTP 429); keep a contact URL or e-mail in `USER_AGENT`, and the cell retries automatically on 429. Total download is about 0.3 GB (music) plus about 20 MB (speech).

In [ ]:
# ---------- automatic download of open speech and music material ----------
import io, os, time, shutil, zipfile, urllib.request, urllib.error
import numpy as np, pandas as pd, soundfile as sf

# Base folder: /content in Google Colab, otherwise ./content next to this notebook (local Jupyter / Kiro).
BASE_DIR = "/content" if os.path.isdir("/content") and "COLAB_RELEASE_TAG" in os.environ else os.path.abspath("content")
print("BASE_DIR =", BASE_DIR)

DATA = dict(
    AUDIO_DIR=os.path.join(BASE_DIR, "audio"),
    DOWNLOAD_DIR=os.path.join(BASE_DIR, "_downloads"),
    MANIFEST=os.path.join(BASE_DIR, "results", "data_manifest.csv"),
    CLIP_S=10.0,                       # identical duration for every clip (s)
    SPEECH_OFFSET_S=0.5,               # skip the first 0.5 s of each EARS file
    EARS_SPEAKERS={"p001": "male", "p002": "female", "p003": "female", "p004": "male"},
    EARS_FILES_PER_SPEAKER=2,          # read sentences, regular style
    MUSIC_OFFSETS_FRAC=(0.30, 0.60),   # excerpt start points as a fraction of the recording length
    # Wikimedia robot policy: the User-Agent must identify the tool and give a contact URL or e-mail,
    # otherwise upload.wikimedia.org answers 403. Replace the URL with your repository or e-mail.
    USER_AGENT="CEPAT-audio-experiment/1.0 (https://github.com/Snivoy; academic research) python-urllib/3",
)
EARS_ZIP = "https://github.com/facebookresearch/ears_dataset/releases/download/dataset/{spk}.zip"
WM = "https://upload.wikimedia.org/wikipedia/commons/"
MUSIC = [  # all verified as 48 kHz / 24 bit / 2 ch FLAC, public domain (Musopen)
    ("beethoven_coriolan", WM + "e/e1/Beethoven_-_Coriolan_Overture%2C_Op._62_%28Musopen_Symphony%29.flac"),
    ("borodin_steppes", WM + "c/c6/Borodin_-_In_the_Steppes_of_Central_Asia_%28Musopen_Symphony%29.flac"),
    ("mozart_figaro", WM + "d/d2/Mozart_-_Le_nozze_di_Figaro%2C_K492_-_Overture_%28Musopen_Symphony%29.flac"),
    ("mendelssohn_quartet6", WM + "d/d2/Mendelssohn_-_String_Quartet_No._6_in_F_minor%2C_Op._80_-_I._Allegro_vivace_assai_%28Musopen_String_Quartet%29.flac"),
]


def _open(url, headers=None, tries=6, timeout=300):
    """urlopen with a descriptive User-Agent and exponential back-off on throttling/server errors."""
    for k in range(tries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": DATA["USER_AGENT"], **(headers or {})})
            return urllib.request.urlopen(req, timeout=timeout)
        except urllib.error.HTTPError as e:
            if e.code in (429, 500, 502, 503, 504) and k < tries - 1:
                wait = int(e.headers.get("Retry-After") or 0) or 15 * 2 ** k
                print(f"    HTTP {e.code}; retry in {wait} s")
                time.sleep(wait)
            else:
                raise


def download(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    tmp = path + ".part"
    with _open(url) as r, open(tmp, "wb") as f:
        shutil.copyfileobj(r, f, length=1 << 20)
    os.replace(tmp, path)
    return path


class HttpRangeFile(io.RawIOBase):
    """Seekable read-only view of a remote file (HTTP Range); lets zipfile read single members."""
    def __init__(self, url):
        with _open(url, timeout=60) as r:          # follows the GitHub redirect once
            self.url, self.size = r.geturl(), int(r.headers["Content-Length"])
        self.pos = 0
    def readable(self): return True
    def seekable(self): return True
    def tell(self): return self.pos
    def seek(self, off, whence=0):
        self.pos = {0: off, 1: self.pos + off, 2: self.size + off}[whence]
        return self.pos
    def readinto(self, b):
        if self.pos >= self.size or len(b) == 0:
            return 0
        end = min(self.pos + len(b), self.size) - 1
        with _open(self.url, {"Range": f"bytes={self.pos}-{end}"}) as r:
            data = r.read()
        b[:len(data)] = data
        self.pos += len(data)
        return len(data)


def save_clip(x, fs, subtype, path):
    sf.write(path, x, fs, subtype=subtype if subtype in ("PCM_16", "PCM_24", "PCM_32", "FLOAT", "DOUBLE") else "PCM_24")


os.makedirs(DATA["DOWNLOAD_DIR"], exist_ok=True)
os.makedirs(os.path.dirname(DATA["MANIFEST"]), exist_ok=True)
manifest = []

# ---- speech (EARS) ----
sp_dir = os.path.join(DATA["AUDIO_DIR"], "speech"); os.makedirs(sp_dir, exist_ok=True)
for spk, gender in DATA["EARS_SPEAKERS"].items():
    print(f"EARS {spk} ({gender})")
    url = EARS_ZIP.format(spk=spk)
    z = zipfile.ZipFile(io.BufferedReader(HttpRangeFile(url), buffer_size=1 << 20))
    need = (DATA["SPEECH_OFFSET_S"] + DATA["CLIP_S"]) * 48000 * 4 + 1024        # bytes at 48 kHz / 32 bit
    members = sorted((i for i in z.infolist()
                      if i.filename.endswith("_regular.wav") and "/sentences_" in i.filename and i.file_size >= need),
                     key=lambda i: i.filename)
    for info in members[:DATA["EARS_FILES_PER_SPEAKER"]]:
        out = os.path.join(sp_dir, f"ears_{spk}_{os.path.basename(info.filename)}")
        raw = z.read(info)                                # only this member is transferred
        x, fs = sf.read(io.BytesIO(raw), always_2d=True)
        meta = sf.info(io.BytesIO(raw))
        a = int(DATA["SPEECH_OFFSET_S"] * fs); n = int(DATA["CLIP_S"] * fs)
        save_clip(x[a:a + n], fs, meta.subtype, out)
        manifest.append(dict(category="speech", file=os.path.basename(out), source="EARS " + info.filename,
                             source_url=url, license="CC BY-NC 4.0", speaker_gender=gender,
                             offset_s=DATA["SPEECH_OFFSET_S"], duration_s=n / fs, fs=fs,
                             subtype=meta.subtype, channels=x.shape[1]))
        print("   ", os.path.basename(out), fs, meta.subtype)

# ---- music (Musopen / Wikimedia Commons) ----
mu_dir = os.path.join(DATA["AUDIO_DIR"], "music"); os.makedirs(mu_dir, exist_ok=True)
for name, url in MUSIC:
    print("Musopen", name)
    flac = download(url, os.path.join(DATA["DOWNLOAD_DIR"], name + ".flac"))
    meta = sf.info(flac)
    n = int(DATA["CLIP_S"] * meta.samplerate)
    for k, frac in enumerate(DATA["MUSIC_OFFSETS_FRAC"]):
        a = int(frac * meta.frames)
        x, fs = sf.read(flac, start=a, stop=a + n, always_2d=True)
        out = os.path.join(mu_dir, f"musopen_{name}_{chr(97 + k)}.wav")
        save_clip(x, fs, meta.subtype, out)
        manifest.append(dict(category="music", file=os.path.basename(out), source="Musopen via Wikimedia Commons",
                             source_url=url, license="Public domain", speaker_gender="",
                             offset_s=a / fs, duration_s=n / fs, fs=fs, subtype=meta.subtype,
                             channels=x.shape[1]))
        print("   ", os.path.basename(out), fs, meta.subtype, f"offset {a / fs:.1f} s")
    time.sleep(2)                       # be polite to the Wikimedia servers

man = pd.DataFrame(manifest)
man.to_csv(DATA["MANIFEST"], index=False)
print(man.groupby("category").agg(clips=("file", "size"), fs=("fs", "unique"), subtype=("subtype", "unique"),
                                  duration_s=("duration_s", "mean")))

In [ ]:
import os, glob, json, platform, tempfile
from math import gcd
import numpy as np
import pandas as pd
import scipy
from scipy import signal
import soundfile as sf
import matplotlib
import matplotlib.pyplot as plt

BASE_DIR = "/content" if os.path.isdir("/content") and "COLAB_RELEASE_TAG" in os.environ else os.path.abspath("content")
CONFIG = dict(
    AUDIO_DIR=os.path.join(BASE_DIR, "audio"),     # <category>/<clip>.wav
    OUT_DIR=os.path.join(BASE_DIR, "results"),
    SAMPLING_RATES=[8000, 16000, 22050, 32000, 44100, 48000],   # only rates <= native rate are used
    BIT_DEPTHS_FILE=[8, 16, 24],         # depths with a native WAV container (U8, 16, 24)
    BIT_DEPTHS_SWEEP=[4, 6, 8, 10, 12, 14, 16, 20, 24],         # quantization-only sweep (Series B)
    PEAK_TARGET=0.95,                    # peak normalisation (fraction of full scale)
    EDGE_TRIM_S=0.05,                    # seconds trimmed at both ends to exclude filter transients
    WELCH_NPERSEG=4096,
    ROLLOFF_FRACTION=0.95,               # spectral roll-off point (95 % of spectral power)
    SEED=42,
)
os.makedirs(CONFIG["OUT_DIR"], exist_ok=True)
rng = np.random.default_rng(CONFIG["SEED"])   # reserved (no dither is used by default)

env = dict(python=platform.python_version(), numpy=np.__version__, scipy=scipy.__version__,
           pandas=pd.__version__, soundfile=sf.__version__, libsndfile=sf.__libsndfile_version__,
           matplotlib=matplotlib.__version__, config=CONFIG)
json.dump(env, open(os.path.join(CONFIG["OUT_DIR"], "environment.json"), "w"), indent=2)
print(json.dumps(env, indent=2))

In [ ]:
# ---------- core functions ----------
def load_clip(path, peak=CONFIG["PEAK_TARGET"]):
    x, fs = sf.read(path, always_2d=True, dtype="float64")
    x = x.mean(axis=1)                      # mono down-mix
    m = np.max(np.abs(x))
    if m > 0:
        x = x * (peak / m)
    return x, fs

def resample(x, fs_in, fs_out):
    if fs_in == fs_out:
        return x.copy()
    g = gcd(int(fs_in), int(fs_out))
    return signal.resample_poly(x, int(fs_out // g), int(fs_in // g))   # SciPy default Kaiser FIR

def quantize(x, B):
    """Uniform mid-tread PCM quantizer, range [-1, 1), no dither."""
    delta = 2.0 ** (1 - B)
    lo, hi = -2 ** (B - 1), 2 ** (B - 1) - 1
    k = np.round(x / delta)
    clipped = int(np.sum((k < lo) | (k > hi)))
    k = np.clip(k, lo, hi)
    return k * delta, delta, clipped

def align(ref, est, fs, trim_s=CONFIG["EDGE_TRIM_S"]):
    n = min(len(ref), len(est)); t = int(trim_s * fs)
    return ref[t:n - t], est[t:n - t]

def snr_db(ref, est):
    err = np.sum((ref - est) ** 2)
    return np.inf if err == 0 else 10 * np.log10(np.sum(ref ** 2) / err)

def rmse(ref, est):
    return float(np.sqrt(np.mean((ref - est) ** 2)))

def psd(x, fs):
    return signal.welch(x, fs=fs, window="hann", nperseg=min(CONFIG["WELCH_NPERSEG"], len(x)),
                        noverlap=None, detrend=False, scaling="spectrum")

def centroid_rolloff(x, fs, frac=CONFIG["ROLLOFF_FRACTION"]):
    f, P = psd(x, fs)
    tot = P.sum()
    c = float((f * P).sum() / tot)
    r = float(f[np.searchsorted(np.cumsum(P), frac * tot)])
    return c, r

def energy_above(x, fs, f_cut):
    f, P = psd(x, fs)
    return float(P[f > f_cut].sum() / P.sum())     # fraction of reference power above f_cut

SUBTYPE = {8: "PCM_U8", 16: "PCM_16", 24: "PCM_24"}

def write_and_measure(xq, fs, B):
    """Write xq as WAV, return (actual bytes, max |readback - xq|)."""
    with tempfile.TemporaryDirectory() as d:
        p = os.path.join(d, "t.wav")
        sf.write(p, xq, int(fs), subtype=SUBTYPE[B])
        size = os.path.getsize(p)
        back, _ = sf.read(p, dtype="float64")
    return size, float(np.max(np.abs(back - xq)))

def theo_bytes(n_samples, B, channels=1):
    return n_samples * B * channels / 8.0

In [ ]:
# ---------- run experiment ----------
rows = []
paths = sorted(glob.glob(os.path.join(CONFIG["AUDIO_DIR"], "**", "*.wav"), recursive=True))
assert paths, f"No WAV files found in {CONFIG['AUDIO_DIR']}"
example = {}

for p in paths:
    cat = os.path.basename(os.path.dirname(p)); name = os.path.basename(p)
    x_ref, fs_ref = load_clip(p)
    P_x = float(np.mean(x_ref ** 2))
    c_ref, r_ref = centroid_rolloff(x_ref, fs_ref)
    rates = [r for r in CONFIG["SAMPLING_RATES"] if r <= fs_ref]
    if fs_ref not in rates:
        rates.append(fs_ref)
    base = dict(clip=name, category=cat, fs_ref=fs_ref, duration_s=len(x_ref) / fs_ref,
                centroid_ref_hz=c_ref, rolloff_ref_hz=r_ref)

    def metrics(xr_est, fs_k, B, series, extra):
        a, b = align(x_ref, xr_est, fs_ref)
        c, r = centroid_rolloff(b, fs_ref)
        row = dict(base, series=series, fs_k=fs_k, B=B, snr_db=snr_db(a, b), rmse=rmse(a, b),
                   rmse_dbfs=20 * np.log10(max(rmse(a, b), 1e-300)),
                   centroid_est_hz=c, rolloff_est_hz=r, d_centroid_hz=c - c_ref, d_rolloff_hz=r - r_ref)
        row.update(extra)
        rows.append(row)

    # Series A: sampling rate only
    for fs_k in rates:
        xk = resample(x_ref, fs_ref, fs_k)
        xr = resample(xk, fs_k, fs_ref)
        el = energy_above(x_ref, fs_ref, fs_k / 2) if fs_k < fs_ref else 0.0
        bound = np.inf if el == 0 else -10 * np.log10(el)
        metrics(xr, fs_k, np.nan, "A", dict(e_loss_pct=100 * el, snr_bound_ideal_db=bound))
        if fs_k in (rates[0], rates[len(rates) // 2], fs_ref):
            example.setdefault(name, {})[fs_k] = xr

    # Series B: bit depth only (native rate)
    for B in CONFIG["BIT_DEPTHS_SWEEP"]:
        xq, delta, clipped = quantize(x_ref, B)
        e_q = x_ref - xq
        metrics(xq, fs_ref, B, "B", dict(
            delta=delta, clipped=clipped, snr_q_db=snr_db(x_ref, xq),
            snr_q_theory_db=10 * np.log10(P_x / (delta ** 2 / 12)),
            qerr_rms=float(np.sqrt(np.mean(e_q ** 2))), qerr_rms_theory=delta / np.sqrt(12),
            size_th_bytes=theo_bytes(len(x_ref), B)))

    # Series C: full factorial
    for fs_k in rates:
        xk = resample(x_ref, fs_ref, fs_k)
        for B in CONFIG["BIT_DEPTHS_FILE"]:
            xq, delta, clipped = quantize(xk, B)
            xr = resample(xq, fs_k, fs_ref)
            size_act, rb = write_and_measure(xq, fs_k, B)
            size_th = theo_bytes(len(xk), B)
            e_q = xk - xq
            el = energy_above(x_ref, fs_ref, fs_k / 2) if fs_k < fs_ref else 0.0
            metrics(xr, fs_k, B, "C", dict(
                delta=delta, clipped=clipped, snr_q_db=snr_db(xk, xq),
                qerr_rms=float(np.sqrt(np.mean(e_q ** 2))), e_loss_pct=100 * el,
                size_th_bytes=size_th, size_act_bytes=size_act,
                size_dev_pct=100 * (size_act - size_th) / size_th, readback_max_abs_diff=rb))

df = pd.DataFrame(rows)
df.to_csv(os.path.join(CONFIG["OUT_DIR"], "results_raw.csv"), index=False)
print(df.groupby("series").size())

In [ ]:
# ---------- aggregation ----------
# Series A at fs_k == fs_ref is an identity operation (resample() returns a copy), so the error is exactly
# zero: SNR = +inf, SNR_BL = +inf and RMSE = 0 (20*log10 -> -inf). Infinite values are kept in
# results_raw.csv (they are correct), but they are excluded from mean/std here and the rows are flagged,
# otherwise pandas returns inf for the mean and NaN for the std of the whole group.
df["is_identity"] = (df.series == "A") & (df.fs_k == df.fs_ref)
dfa = df.replace([np.inf, -np.inf], np.nan)
dfa.loc[dfa.rmse == 0, "rmse_dbfs"] = np.nan

num = dfa.select_dtypes("number").columns.difference(["fs_ref"])
keys = ["series", "category", "fs_k", "B"]
agg = dfa.groupby(keys, dropna=False)[list(num)].agg(["mean", "std"])
agg.columns = ["_".join(c) for c in agg.columns]
agg = agg.join(dfa.groupby(keys, dropna=False).agg(n_clips=("clip", "size"),
                                                   n_identity=("is_identity", "sum"))).reset_index()
agg["note"] = np.where(agg.n_identity > 0,
                       "identity (fs_k = fs_ref): zero error, SNR = +inf, excluded from mean", "")
agg = agg.round(6)
agg.to_csv(os.path.join(CONFIG["OUT_DIR"], "results_summary.csv"), index=False, na_rep="")

# Slope of quantization SNR versus bit depth (Series B, least squares over B in the linear region)
sb = df[df.series == "B"].groupby("B")["snr_q_db"].mean().reset_index()
lin = sb[(sb.B >= 8) & (sb.B <= 16)]
slope = np.polyfit(lin.B, lin.snr_q_db, 1)[0]
print(f"Measured slope of SNR_q vs B (8-16 bit): {slope:.2f} dB/bit")

# Pareto frontier on Series C (size minimised, SNR maximised) using category-pooled means
c = (df[df.series == "C"].groupby(["fs_k", "B"])[["snr_db", "size_act_bytes"]].mean().reset_index())
c["size_mib"] = c.size_act_bytes / 2 ** 20
c = c.sort_values(["size_mib", "snr_db"], ascending=[True, False]).reset_index(drop=True)
best, front = -np.inf, []
for i, r in c.iterrows():
    if r.snr_db > best:
        front.append(i); best = r.snr_db
c["pareto"] = c.index.isin(front)
fr = c[c.pareto].copy()
fr["marginal_db_per_mib"] = fr.snr_db.diff() / fr.size_mib.diff()
c = c.join(fr["marginal_db_per_mib"])
c.to_csv(os.path.join(CONFIG["OUT_DIR"], "series_C_pooled_pareto.csv"), index=False)
print(fr[["fs_k", "B", "size_mib", "snr_db", "marginal_db_per_mib"]])

# Readable Series A view: identity rows shown as text instead of inf
tA = agg[agg.series == "A"][["category", "fs_k", "snr_db_mean", "snr_db_std", "rmse_mean",
                              "e_loss_pct_mean", "snr_bound_ideal_db_mean", "d_centroid_hz_mean",
                              "d_rolloff_hz_mean", "n_clips", "note"]].copy()
for col in ["snr_db_mean", "snr_bound_ideal_db_mean"]:
    tA[col] = tA[col].map(lambda v: "inf (identity)" if pd.isna(v) else f"{v:.2f}")
print(tA.to_string(index=False))

In [ ]:
# ---------- figures ----------
out = CONFIG["OUT_DIR"]
plt.rcParams.update({"font.family": "serif", "font.size": 9})

A = df[df.series == "A"]
fig, ax = plt.subplots(figsize=(3.4, 2.4))
for cat, g in A.groupby("category"):
    m = g.groupby("fs_k")["snr_db"].agg(["mean", "std"]).replace(np.inf, np.nan)
    ax.errorbar(m.index, m["mean"], yerr=m["std"], marker="o", capsize=2, label=cat)
ax.set_xlabel("Sampling rate (Hz)"); ax.set_ylabel("SNR (dB)"); ax.legend(frameon=False); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(os.path.join(out, "fig_A_snr_vs_fs.png"), dpi=300); plt.close(fig)

Bd = df[df.series == "B"]
fig, ax = plt.subplots(figsize=(3.4, 2.4))
m = Bd.groupby("B")[["snr_q_db", "snr_q_theory_db"]].mean()
ax.plot(m.index, m.snr_q_db, "o-", label="measured"); ax.plot(m.index, m.snr_q_theory_db, "s--", label="uniform-noise model")
ax.set_xlabel("Bit depth (bit)"); ax.set_ylabel("SNR (dB)"); ax.legend(frameon=False); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(os.path.join(out, "fig_B_snr_vs_bits.png"), dpi=300); plt.close(fig)

Cc = df[df.series == "C"].groupby(["fs_k", "B"])["snr_db"].mean().unstack("B")
fig, ax = plt.subplots(figsize=(3.4, 2.6))
im = ax.imshow(Cc.values, origin="lower", aspect="auto", cmap="viridis")
ax.set_xticks(range(len(Cc.columns))); ax.set_xticklabels([int(b) for b in Cc.columns]); ax.set_yticks(range(len(Cc.index))); ax.set_yticklabels(Cc.index)
for i in range(Cc.shape[0]):
    for j in range(Cc.shape[1]):
        ax.text(j, i, f"{Cc.values[i, j]:.1f}", ha="center", va="center", color="w", fontsize=7)
ax.set_xlabel("Bit depth (bit)"); ax.set_ylabel("Sampling rate (Hz)"); fig.colorbar(im, label="SNR (dB)")
fig.tight_layout(); fig.savefig(os.path.join(out, "fig_C_heatmap.png"), dpi=300); plt.close(fig)

fig, ax = plt.subplots(figsize=(3.4, 2.6))
ax.scatter(c.size_mib, c.snr_db, c="gray", s=12, label="configuration")
ax.plot(c[c.pareto].size_mib, c[c.pareto].snr_db, "r-o", ms=4, label="Pareto front")
ax.set_xscale("log"); ax.set_xlabel("File size (MiB)"); ax.set_ylabel("SNR (dB)"); ax.legend(frameon=False); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(os.path.join(out, "fig_C_pareto.png"), dpi=300); plt.close(fig)

if example:
    nm = sorted(example)[0]
    x_ref, fs_ref = load_clip(next(p for p in paths if os.path.basename(p) == nm))
    fig, ax = plt.subplots(figsize=(3.4, 2.4))
    f, P = psd(x_ref, fs_ref); ax.semilogy(f, P + 1e-20, label="reference", lw=1.2)
    for k, xr in example[nm].items():
        if k != fs_ref:
            f, P = psd(xr[:len(x_ref)], fs_ref); ax.semilogy(f, P + 1e-20, label=f"{k} Hz", lw=.8)
    ax.set_xlabel("Frequency (Hz)"); ax.set_ylabel("Power spectrum"); ax.legend(frameon=False, fontsize=7); ax.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(os.path.join(out, "fig_spectrum_example.png"), dpi=300); plt.close(fig)
print("Done. Files in", out, sorted(os.listdir(out)))

## 2-D spectrogram comparison (speech vs music)
One representative clip per category; columns show the reference and four degradations taken from Series A, B, and C. Dashed lines mark fs_k/2. Power is in dB relative to the maximum of the reference spectrogram of the same clip, so the colour scale is comparable across columns.

In [ ]:
# ---------- figure: 2-D spectrograms, speech vs music, several sampling rates and bit depths ----------
SPEC = dict(
    CASES=[("Reference", None, None),          # (title, fs_k in Hz or None, B in bit or None)
           ("8 kHz", 8000, None),
           ("22.05 kHz", 22050, None),
           ("8 bit", None, 8),
           ("8 kHz, 8 bit", 8000, 8)],
    EXCERPT_S=4.0, NFFT=2048, HOP=512, DB_RANGE=100.0, CMAP="magma",
)

def degrade(x, fs, fs_k=None, B=None):
    """Same processing chain as Series A/B/C: resample -> quantize -> resample back to fs."""
    y = x
    if fs_k and fs_k < fs:
        y = resample(y, fs, fs_k)
    if B:
        y = quantize(y, B)[0]
    if fs_k and fs_k < fs:
        y = resample(y, fs_k, fs)
    return y[:len(x)]

def stft_power(x, fs):
    f, t, Z = signal.stft(x, fs=fs, window="hann", nperseg=SPEC["NFFT"],
                          noverlap=SPEC["NFFT"] - SPEC["HOP"], boundary=None, padded=False)
    return f, t, np.abs(Z) ** 2

by_cat = {}
for p in paths:
    by_cat.setdefault(os.path.basename(os.path.dirname(p)), []).append(p)
cats = [k for k in ("speech", "music") if k in by_cat] + sorted(k for k in by_cat if k not in ("speech", "music"))

fig, axes = plt.subplots(len(cats), len(SPEC["CASES"]), figsize=(7.2, 1.75 * len(cats) + 0.5),
                         sharex=True, sharey=True, squeeze=False, constrained_layout=True)
for i, cat in enumerate(cats):
    p = sorted(by_cat[cat])[0]
    x, fs = load_clip(p)
    n = min(len(x), int(SPEC["EXCERPT_S"] * fs)); a = (len(x) - n) // 2
    seg = x[a:a + n]
    f, t, P_ref = stft_power(seg, fs)
    ref = P_ref.max()
    for j, (title, fs_k, B) in enumerate(SPEC["CASES"]):
        ax = axes[i, j]
        _, _, P = stft_power(seg if (fs_k is None and B is None) else degrade(seg, fs, fs_k, B), fs)
        S = 10 * np.log10(P / ref + 1e-20)
        im = ax.pcolormesh(t, f / 1e3, S, shading="auto", cmap=SPEC["CMAP"],
                           vmin=-SPEC["DB_RANGE"], vmax=0, rasterized=True)
        if fs_k:
            ax.axhline(fs_k / 2e3, color="w", ls="--", lw=0.6)
        if i == 0:
            ax.set_title(title, fontsize=8)
        if j == 0:
            ax.set_ylabel(f"{cat.capitalize()}\nFrequency (kHz)")
        if i == len(cats) - 1:
            ax.set_xlabel("Time (s)")
    print(f"{cat}: {os.path.basename(p)}, fs = {fs} Hz, excerpt {a / fs:.2f}-{(a + n) / fs:.2f} s")
fig.colorbar(im, ax=axes, shrink=0.9, pad=0.01, label="Power (dB re. max of reference)")
fig.savefig(os.path.join(out, "fig_spectrogram_speech_music.png"), dpi=300)
plt.show()

## Sensitivity check: sharper resampling filter (Series A)
SciPy's default `resample_poly` filter (Kaiser, beta = 5, half-length 10*max(p, q)) has a transition band that attenuates components just below fs_k/2. This cell repeats Series A with a Kaiser FIR five times longer (half-length 50*max(p, q), beta = 12) and compares the SNR with the default filter and with the ideal bound SNR_BL. Output: `sensitivity_filter.csv`.

In [ ]:
# ---------- sensitivity check: sharper anti-aliasing filter (Series A) ----------
def resample_sharp(x, fs_in, fs_out, mult=50, beta=12.0):
    """Polyphase resampling with a Kaiser FIR 5x longer than the SciPy default (half-length 50*max(p,q), beta=12)."""
    if fs_in == fs_out:
        return x.copy()
    g = gcd(int(fs_in), int(fs_out)); up, down = int(fs_out // g), int(fs_in // g)
    m = max(up, down)
    h = signal.firwin(2 * mult * m + 1, 1.0 / m, window=("kaiser", beta))   # resample_poly scales h by up
    return signal.resample_poly(x, up, down, window=h)

sens = []
for p in paths:
    x_ref, fs_ref = load_clip(p)
    for fs_k in [r for r in CONFIG["SAMPLING_RATES"] if r < fs_ref]:
        a_, b_ = align(x_ref, resample_sharp(resample_sharp(x_ref, fs_ref, fs_k), fs_k, fs_ref), fs_ref)
        sens.append(dict(category=os.path.basename(os.path.dirname(p)), fs_k=fs_k, snr_sharp_db=snr_db(a_, b_)))
sens = pd.DataFrame(sens).groupby(["category", "fs_k"]).snr_sharp_db.mean()
base_ = (df[(df.series == "A") & (df.fs_k < df.fs_ref)]
         .groupby(["category", "fs_k"])[["snr_db", "snr_bound_ideal_db"]].mean())
cmp_ = base_.join(sens)
cmp_["delta_db"] = cmp_.snr_sharp_db - cmp_.snr_db
cmp_["gap_to_bound_db"] = cmp_.snr_bound_ideal_db - cmp_.snr_sharp_db
cmp_.round(4).to_csv(os.path.join(CONFIG["OUT_DIR"], "sensitivity_filter.csv"))
print(cmp_.round(2))